# Lab 2: Detecting Suspicious Login Activity Using Machine Learning

**ENSP355 — AI in Cyber Security Lab (2026–27)**

This notebook answers Q1–Q6 using the included synthetic teaching CSV. If your instructor supplies the official CSV, replace `data/simple_login_activity.csv` first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

DATA = Path("../data/simple_login_activity.csv")
OUT = Path("../outputs")
OUT.mkdir(exist_ok=True)


## Q1. Dataset inspection and cleaning

**Aim:** Identify input columns and target label, remove duplicates and replace missing numeric values with medians.

In [ ]:
df = pd.read_csv(DATA)
display(df.head())
print(df.info())
print("Missing values before cleaning:")
display(df.isna().sum())
print("Duplicate rows before cleaning:", df.duplicated().sum())

df = df.drop_duplicates().copy()
for col in ["failed_logins", "download_mb"]:
    df[col] = df[col].fillna(df[col].median())

print("Missing values after cleaning:", int(df.isna().sum().sum()))
print("Duplicate rows after cleaning:", int(df.duplicated().sum()))
print("Final rows:", len(df))
display(df.head())
features = ["failed_logins", "login_hour", "new_device", "download_mb"]
print("Input features:", features)
print("Target label: label")


## Q2. Training and testing split

**Aim:** Use 75% of the data for training and 25% for testing, with class proportions preserved.

In [ ]:
X = df[features]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Testing on training data is misleading because the model has already seen those records and the score does not estimate performance on unseen activity.")


## Q3. Decision Tree training and evaluation

**Aim:** Train a depth-4 Decision Tree and interpret accuracy and its confusion matrix.

In [ ]:
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")
labels = ["Normal", "Suspicious"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
print("Confusion matrix (rows=actual, columns=predicted):")
print(cm)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=labels)
plt.title("Decision Tree Confusion Matrix")
plt.show()


**Interpretation:** The top-right cell counts false positives (Normal predicted as Suspicious); the bottom-left cell counts false negatives (Suspicious predicted as Normal). Less than 100% accuracy is realistic because login patterns can overlap and the training sample is limited.

## Q4. K-Means clustering

**Aim:** Group records by numeric similarity without using the known labels to fit K-Means.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)
cluster_summary = df.groupby("cluster")[features].mean().round(2)
label_summary = pd.crosstab(df["cluster"], df["label"])
display(cluster_summary)
display(label_summary)
cluster_summary.to_csv(OUT / "cluster_summary.csv")
label_summary.to_csv(OUT / "cluster_vs_label.csv")


**Interpretation:** Compare cluster means and label counts. The cluster with relatively higher failed logins, more new-device use, unusual login hours and larger downloads can be described as more suspicious-looking. Cluster IDs themselves have no fixed meaning.

## Q5. Feature importance and new-record prediction

In [ ]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)
display(importance)
importance.plot(kind="bar", x="feature", y="importance", legend=False, title="Decision Tree Feature Importance")
plt.ylabel("Importance")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()
importance.to_csv(OUT / "feature_importance.csv", index=False)

new_records = pd.DataFrame([
    {"failed_logins": 0, "login_hour": 10, "new_device": 0, "download_mb": 45},
    {"failed_logins": 8, "login_hour": 2, "new_device": 1, "download_mb": 180},
    {"failed_logins": 2, "login_hour": 23, "new_device": 1, "download_mb": 900},
])
new_records["prediction"] = model.predict(new_records[features])
display(new_records)
new_records.to_csv(OUT / "new_record_predictions.csv", index=False)


**Prediction notes:** Record 1 has no failed attempts, a familiar daytime hour and a known device, so it is expected to look normal. Record 2 combines many failed attempts, a very late hour and a new device. Record 3 combines late-night access, a new device and a very large download; the model's actual output is shown in the table.

## Q6. False positives, false negatives and reflection

- **False positive:** legitimate late-night work on a new device is marked Suspicious. This can waste analyst time and inconvenience a user.
- **False negative:** a disguised attack with few obvious indicators is marked Normal. This can allow harmful activity to go unnoticed.
- **Limitation:** the dataset is small and synthetic; it cannot represent the full variety of real login behaviour or attacker techniques.
- **Human review:** an analyst should review high-risk alerts and unusual combinations before account restrictions or incident conclusions.

In [ ]:
df.to_csv(OUT / "Lab2_Login_ML_Result.csv", index=False)
print("Saved:", OUT / "Lab2_Login_ML_Result.csv")
